# 🏗️ ราคามาตรฐานค่าก่อสร้างอาคาร รายจังหวัด — 2567

วิเคราะห์**ราคาค่าก่อสร้างต่อตารางเมตร (บาท/ตร.ม.)** ของอาคาร 69 ประเภท ใน 77 จังหวัด  
ที่มา: `construct_all_20240805.csv` (บัญชีราคามาตรฐานค่าก่อสร้าง)

- **Treemap** — หมวดอาคาร → ประเภท (ตามราคา)
- **แผนที่** — ราคาค่าก่อสร้างเฉลี่ยรายจังหวัด
- **K-Means** — จัดกลุ่มจังหวัดตามโครงสร้างราคา
- **Ridge Regression** — ประเมินราคาค่าก่อสร้างจากประเภทอาคาร + ภูมิภาค

In [ ]:
# Google Colab — run this cell first to install dependencies
!pip install pandas plotly scikit-learn -q

In [ ]:
import pandas as pd, numpy as np, os, json, urllib.request
import plotly.graph_objects as go
import plotly.express as px
from plotly.io import to_html
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv('construct_all_20240805.csv', encoding='utf-8-sig')
df.columns = df.columns.str.strip()
df = df.dropna(subset=['NAME_CONSTR','CHANGWAT_NAME','PRICE_CONSTR']).reset_index(drop=True)
df['PRICE_CONSTR'] = df['PRICE_CONSTR'].astype(float)
df['NAME_CONSTR']  = df['NAME_CONSTR'].astype(str).str.strip()
df['CHANGWAT_NAME']= df['CHANGWAT_NAME'].astype(str).str.strip()

# building category from the ID prefix (1xx บ้าน, 2xx ทาวน์เฮาส์, ... 6xx งานภายนอก)
CATNAME = {'1':'บ้านพักอาศัย','2':'บ้านแถว/ทาวน์เฮาส์','3':'ห้องแถวไม้',
           '4':'ตึกแถว','5':'อาคารเฉพาะกิจ','6':'งานภายนอก/อื่นๆ'}
df['cat']      = df['ID_CONSTR'].astype(str).str[0]
df['cat_name'] = df['cat'].map(CATNAME)

n_types = df['NAME_CONSTR'].nunique()
n_prov  = df['CHANGWAT_NAME'].nunique()
avg_p   = df['PRICE_CONSTR'].mean()
min_p, max_p = df['PRICE_CONSTR'].min(), df['PRICE_CONSTR'].max()
print(f'Rows           : {len(df):,}')
print(f'Building types : {n_types}   Provinces: {n_prov}')
print(f'Price ฿/ตร.ม.  : avg {avg_p:,.0f}  (min {min_p:,.0f} – max {max_p:,.0f})')
df.head(3)

In [ ]:
PALETTE = [
    '#bc6c25','#3d405b','#dda15e','#606c38','#e07a5f',
    '#81b29a','#e9c46a','#a44a3f','#283618','#f2cc8f',
    '#9c6644','#7f5539','#b08968','#ddb892','#495057',
]
CLUSTER_COLORS = ['#bc6c25','#3d405b','#e07a5f','#606c38','#81b29a']
CAT_COLORS = {'บ้านพักอาศัย':'#e07a5f','บ้านแถว/ทาวน์เฮาส์':'#f2cc8f','ห้องแถวไม้':'#dda15e',
              'ตึกแถว':'#bc6c25','อาคารเฉพาะกิจ':'#3d405b','งานภายนอก/อื่นๆ':'#81b29a'}
CAT_ORDER = ['บ้านพักอาศัย','บ้านแถว/ทาวน์เฮาส์','ห้องแถวไม้','ตึกแถว','อาคารเฉพาะกิจ','งานภายนอก/อื่นๆ']
print('Palette ready.')

## Chart 1 — Treemap: หมวดอาคาร → ประเภท

ขนาด & สี = ราคาค่าก่อสร้างเฉลี่ย (บาท/ตร.ม.) · คลิกเพื่อ zoom

In [ ]:
tp = (df.groupby(['cat_name','NAME_CONSTR'])['PRICE_CONSTR'].mean()
        .reset_index().rename(columns={'PRICE_CONSTR':'price'}))
tp['price'] = tp['price'].round(0)
fig_tree = px.treemap(tp, path=[px.Constant('อาคารทั้งหมด'),'cat_name','NAME_CONSTR'],
    values='price', color='price', color_continuous_scale='YlOrBr',
    custom_data=['price'])
fig_tree.update_traces(
    texttemplate='<b>%{label}</b><br>%{color:,.0f} ฿/ตร.ม.',
    hovertemplate='<b>%{label}</b><br>ราคาเฉลี่ย: %{color:,.0f} ฿/ตร.ม.<extra></extra>',
    marker=dict(line=dict(color='white',width=1)))
fig_tree.update_layout(title='ราคาค่าก่อสร้างอาคาร: หมวด → ประเภท (บาท/ตร.ม.)',
    height=600, margin=dict(t=50,l=4,r=4,b=4), coloraxis_colorbar=dict(title='฿/ตร.ม.'))
fig_tree.show()

## Chart 2 — แผนที่ประเทศไทย: ราคาค่าก่อสร้างเฉลี่ยรายจังหวัด

Choropleth · สีเข้ม = ก่อสร้างแพงกว่า (เฉลี่ยทุกประเภทอาคาร)

In [ ]:
def _load_geojson(local, url):
    try:    return json.load(open(local, encoding='utf-8'))
    except Exception:
        with urllib.request.urlopen(url, timeout=30) as r: return json.load(r)
geojson = _load_geojson('th_provinces.geojson',
    'https://raw.githubusercontent.com/chingchai/OpenGISData-Thailand/master/provinces.geojson')
prov_region = {f['properties']['pro_th']: f['properties'].get('reg_royin','') for f in geojson['features']}
df['region'] = df['CHANGWAT_NAME'].map(prov_region).astype(str)

pmap = df.groupby('CHANGWAT_NAME')['PRICE_CONSTR'].mean().reset_index()
pmap['PRICE_CONSTR'] = pmap['PRICE_CONSTR'].round(0)
fig_map = px.choropleth(pmap, geojson=geojson, featureidkey='properties.pro_th',
    locations='CHANGWAT_NAME', color='PRICE_CONSTR', color_continuous_scale='YlOrBr',
    custom_data=['CHANGWAT_NAME','PRICE_CONSTR'])
fig_map.update_traces(hovertemplate='<b>%{customdata[0]}</b><br>ราคาเฉลี่ย: %{customdata[1]:,.0f} ฿/ตร.ม.<extra></extra>')
fig_map.update_geos(fitbounds='locations', visible=False)
fig_map.update_layout(title='ราคาค่าก่อสร้างเฉลี่ยรายจังหวัด (บาท/ตร.ม.)',
    height=640, margin=dict(t=50,l=0,r=0,b=0), coloraxis_colorbar=dict(title='฿/ตร.ม.'))
fig_map.show()
print('แพงสุด:', pmap.nlargest(3,'PRICE_CONSTR').set_index('CHANGWAT_NAME')['PRICE_CONSTR'].to_dict())
print('ถูกสุด:', pmap.nsmallest(3,'PRICE_CONSTR').set_index('CHANGWAT_NAME')['PRICE_CONSTR'].to_dict())

## Chart 3 — การกระจายราคาตามหมวดอาคาร

In [ ]:
fig_box = px.box(df, x='cat_name', y='PRICE_CONSTR', color='cat_name',
    category_orders={'cat_name':CAT_ORDER}, color_discrete_map=CAT_COLORS, points=False)
fig_box.update_layout(title='การกระจายราคาค่าก่อสร้างตามหมวดอาคาร',
    xaxis_title='หมวดอาคาร', yaxis_title='ราคา (บาท/ตร.ม.)', height=440,
    plot_bgcolor='white', showlegend=False, margin=dict(t=55,b=80,l=65,r=20))
fig_box.update_xaxes(tickangle=-20); fig_box.update_yaxes(showgrid=True,gridcolor='#e9ecef')
fig_box.show()

## Chart 4 — Top 15 ประเภทอาคารที่ก่อสร้างแพงที่สุด

In [ ]:
top = df.groupby('NAME_CONSTR')['PRICE_CONSTR'].mean().sort_values(ascending=False).head(15).reset_index()
fig_bar = go.Figure(go.Bar(
    x=top['PRICE_CONSTR'][::-1].values, y=top['NAME_CONSTR'][::-1].values, orientation='h',
    marker=dict(color=top['PRICE_CONSTR'][::-1].values, colorscale='YlOrBr', line=dict(color='white',width=1)),
    text=top['PRICE_CONSTR'][::-1].apply(lambda v:f'{v:,.0f}'), textposition='outside',
    hovertemplate='<b>%{y}</b><br>%{x:,.0f} ฿/ตร.ม.<extra></extra>'))
fig_bar.update_layout(title='15 ประเภทอาคารที่ราคาค่าก่อสร้างสูงสุด (บาท/ตร.ม.)',
    xaxis_title='ราคาเฉลี่ย (บาท/ตร.ม.)', height=520, plot_bgcolor='white',
    margin=dict(t=55,b=50,l=260,r=80))
fig_bar.update_xaxes(showgrid=True,gridcolor='#e9ecef')
fig_bar.show()

## K-Means Clustering — จัดกลุ่มจังหวัดตามโครงสร้างราคา

**Feature (7):** ราคาเฉลี่ยของแต่ละหมวดอาคาร 6 หมวด + ราคาเฉลี่ยรวม → StandardScaler → KMeans → PCA

In [ ]:
piv = df.pivot_table(index='CHANGWAT_NAME', columns='cat_name', values='PRICE_CONSTR', aggfunc='mean')
piv = piv[CAT_ORDER]
piv['ราคาเฉลี่ยรวม'] = df.groupby('CHANGWAT_NAME')['PRICE_CONSTR'].mean()
X = StandardScaler().fit_transform(piv.values)
k = 4
km = KMeans(n_clusters=k, random_state=42, n_init=10)
clab = km.fit_predict(X)
pca = PCA(n_components=2, random_state=42); coords = pca.fit_transform(X)
var_exp = pca.explained_variance_ratio_
pca_df = pd.DataFrame({'PC1':coords[:,0],'PC2':coords[:,1],'prov':piv.index,
    'cluster':clab.astype(str),'avg_price':piv['ราคาเฉลี่ยรวม'].values,
    'region':[prov_region.get(p,'') for p in piv.index]})
prof = piv.copy(); prof['cluster']=clab
clu_means = prof.groupby('cluster')[CAT_ORDER].mean()
clu_price = prof.groupby('cluster')['ราคาเฉลี่ยรวม'].mean()
print(f'k={k}  PC1={var_exp[0]:.1%} PC2={var_exp[1]:.1%}')
for c in sorted(pca_df['cluster'].unique(),key=int):
    s=pca_df[pca_df['cluster']==c]
    print(f"  กลุ่ม {c}: {len(s):2d} จว. · เฉลี่ย {s['avg_price'].mean():,.0f} ฿/ตร.ม. · {', '.join(sorted(s['prov'])[:4])}...")

### Chart 5 — PCA Scatter: K-Means กลุ่มจังหวัด

ขนาดจุด = ราคาเฉลี่ย · สี = กลุ่ม

In [ ]:
fig_pca = go.Figure()
for c in sorted(pca_df['cluster'].unique(),key=int):
    sub=pca_df[pca_df['cluster']==c]
    fig_pca.add_trace(go.Scatter(x=sub['PC1'],y=sub['PC2'],mode='markers+text',name=f'กลุ่ม {c}',
        marker=dict(size=(sub['avg_price']/sub['avg_price'].max()*24+8),
                    color=CLUSTER_COLORS[int(c)%len(CLUSTER_COLORS)],opacity=.85,line=dict(color='white',width=1)),
        text=sub['prov'],textposition='top center',textfont=dict(size=8),
        customdata=sub[['prov','avg_price','region']].values,
        hovertemplate='<b>%{customdata[0]}</b><br>ราคาเฉลี่ย: %{customdata[1]:,.0f} ฿/ตร.ม.<br>ภาค: %{customdata[2]}<extra></extra>'))
fig_pca.update_layout(title=(f'K-Means (k={k}) — PCA Projection | 7 features<br>'
    f'<sup>PC1={var_exp[0]:.1%}, PC2={var_exp[1]:.1%} ของความแปรปรวน</sup>'),
    xaxis_title=f'PC1 ({var_exp[0]:.1%})', yaxis_title=f'PC2 ({var_exp[1]:.1%})',
    height=560, plot_bgcolor='white', legend=dict(title='<b>กลุ่ม</b>'), margin=dict(t=75,b=50,l=65,r=20))
fig_pca.update_xaxes(showgrid=True,gridcolor='#e9ecef',zeroline=True,zerolinecolor='#dee2e6')
fig_pca.update_yaxes(showgrid=True,gridcolor='#e9ecef',zeroline=True,zerolinecolor='#dee2e6')
fig_pca.show()

### Chart 6 — Cluster Profiles: ราคาแต่ละหมวดของแต่ละกลุ่ม

In [ ]:
fig_prof = go.Figure()
for i,cat in enumerate(CAT_ORDER):
    fig_prof.add_trace(go.Bar(name=cat, x=[f'กลุ่ม {c}' for c in clu_means.index], y=clu_means[cat].values,
        marker_color=CAT_COLORS[cat], hovertemplate=f'<b>{cat}</b><br>%{{y:,.0f}} ฿/ตร.ม.<extra></extra>'))
fig_prof.update_layout(barmode='group', title='ราคาค่าก่อสร้างเฉลี่ยตามหมวด แยกตามกลุ่ม K-Means',
    xaxis_title='กลุ่ม', yaxis_title='ราคา (บาท/ตร.ม.)', height=460, plot_bgcolor='white',
    legend=dict(title='หมวดอาคาร',font_size=10,orientation='h',y=-0.18), margin=dict(t=55,b=90,l=65,r=20))
fig_prof.update_yaxes(showgrid=True,gridcolor='#e9ecef')
fig_prof.show()

## 🔮 Predictive Model — ประเมินราคาค่าก่อสร้าง (Ridge Regression)

**โมเดล:** Ridge Regression ทำนาย*ราคาค่าก่อสร้าง (บาท/ตร.ม.)* จาก **ประเภทอาคาร (69) + ภูมิภาค (6)**  
ประเมินด้วย 5-fold cross-validation → แสดง *predicted vs actual* พร้อม R² และ MAE  
ประเภทอาคารเป็นตัวกำหนดราคาหลัก · ภูมิภาคปรับตามต้นทุนพื้นที่

In [ ]:
Xr = pd.concat([pd.get_dummies(df['NAME_CONSTR'],prefix='t'),
                pd.get_dummies(df['region'],prefix='r')], axis=1)
yr = df['PRICE_CONSTR'].values
model = Ridge(alpha=1.0)
yhat = cross_val_predict(model, Xr, yr, cv=KFold(5,shuffle=True,random_state=42))
r2 = r2_score(yr,yhat); mae = mean_absolute_error(yr,yhat)
model.fit(Xr,yr)
lim = max(yr.max(),yhat.max())*1.02
samp = np.random.RandomState(42).choice(len(yr), 1600, replace=False)
fig_reg = go.Figure()
fig_reg.add_trace(go.Scatter(x=[0,lim],y=[0,lim],mode='lines',name='ทำนายตรงจริง',
    line=dict(color='#adb5bd',dash='dash'),hoverinfo='skip'))
fig_reg.add_trace(go.Scatter(x=yr[samp],y=yhat[samp],mode='markers',name='อาคาร',
    marker=dict(size=5,color=df['cat'].astype(int).values[samp],colorscale='YlOrBr',opacity=.45,
                line=dict(width=0)),
    customdata=np.stack([df['NAME_CONSTR'].values[samp],df['CHANGWAT_NAME'].values[samp],
        yr[samp],yhat[samp]],axis=1),
    hovertemplate='<b>%{customdata[0]}</b> · %{customdata[1]}<br>จริง: %{customdata[2]:,.0f}<br>ทำนาย: %{customdata[3]:,.0f}<extra></extra>'))
fig_reg.update_layout(title=f'Predicted vs Actual — Ridge (5-fold CV) · R²={r2:.3f}, MAE={mae:,.0f} ฿/ตร.ม.',
    xaxis_title='ราคาจริง (฿/ตร.ม.)', yaxis_title='ราคาที่ทำนาย (฿/ตร.ม.)',
    height=560, plot_bgcolor='white', showlegend=False, margin=dict(t=55,b=50,l=70,r=20))
fig_reg.update_xaxes(showgrid=True,gridcolor='#e9ecef'); fig_reg.update_yaxes(showgrid=True,gridcolor='#e9ecef')
fig_reg.show()
print(f'Ridge R²={r2:.3f}  MAE={mae:,.0f} ฿/ตร.ม.')

## Export → construct_viz.html

In [ ]:
CSS = """
*{box-sizing:border-box}
body{font-family:'Sarabun','Segoe UI',sans-serif;background:#f4f1ec;margin:0;padding:0;color:#212529}
.page{max-width:1140px;margin:0 auto;padding:32px 24px 64px}
.hero{background:linear-gradient(135deg,#283618 0%,#3d405b 30%,#bc6c25 72%,#e9c46a 100%);border-radius:16px;padding:40px 48px;margin-bottom:24px;color:white;position:relative;overflow:hidden}
.hero::before{content:'🏗️';position:absolute;right:38px;top:14px;font-size:96px;opacity:.14;line-height:1}
.hero h1{margin:0 0 6px;font-size:2.0em;font-weight:700;letter-spacing:-.5px}
.hero p{margin:0;opacity:.8;font-size:1em}
.badge{display:inline-block;background:rgba(255,255,255,.15);border:1px solid rgba(255,255,255,.28);border-radius:20px;padding:3px 13px;font-size:.83em;margin:12px 6px 0 0}
.stats{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin-bottom:24px}
.stat-card{background:white;border-radius:12px;padding:20px 22px;box-shadow:0 1px 5px rgba(0,0,0,.08);border-left:4px solid var(--ac)}
.stat-card .num{font-size:1.9em;font-weight:700;color:var(--ac);line-height:1.1}
.stat-card .lbl{font-size:.84em;color:#6c757d;margin-top:5px}
.card{background:white;border-radius:12px;box-shadow:0 1px 5px rgba(0,0,0,.08);margin-bottom:24px;overflow:hidden}
.card-header{padding:16px 22px 0;font-size:.78em;font-weight:700;text-transform:uppercase;letter-spacing:.07em;color:#6c757d}
.grid-2{display:grid;grid-template-columns:1fr 1fr;gap:24px}
.note{font-size:.82em;color:#6c757d;background:#fff;border-left:4px solid #bc6c25;border-radius:8px;padding:12px 18px;margin-bottom:24px;box-shadow:0 1px 5px rgba(0,0,0,.06)}
@media(max-width:768px){.stats{grid-template-columns:1fr 1fr}.grid-2{grid-template-columns:1fr}.hero h1{font-size:1.4em}.hero::before{display:none}}
@keyframes brickFall{0%{top:-60px;opacity:1}100%{top:110vh;opacity:0}}
@keyframes popIn{0%{opacity:0;transform:translate(-50%,-50%) scale(.05)}60%{transform:translate(-50%,-50%) scale(1.2)}80%{transform:translate(-50%,-50%) scale(.95)}100%{opacity:1;transform:translate(-50%,-50%) scale(1)}}
@keyframes fadeOut{to{opacity:0;transform:translate(-50%,-50%) scale(.6)}}
@keyframes glowPulse{0%,100%{box-shadow:0 0 40px rgba(188,108,37,.7),0 0 80px rgba(233,196,102,.4)}50%{box-shadow:0 0 80px rgba(188,108,37,1),0 0 160px rgba(233,196,102,.7)}}
@keyframes truckRun{0%{left:-90px}100%{left:110vw}}
@keyframes amberTint{0%{opacity:0}50%{opacity:.15}100%{opacity:0}}
"""

BUILD_JS = r"""<script>
(function(){
  var busy=false;
  function mk(css,html){var e=document.createElement('div');e.style.cssText=css;if(html!==undefined)e.innerHTML=html;return e;}
  function go(){
    if(busy)return; busy=true;
    var trash=[]; function add(e){document.body.appendChild(e);trash.push(e);return e;}
    var ICONS=['🧱','🏗️','🔨','🪵','⚙️','📐'];
    for(var i=0;i<34;i++){(function(idx){setTimeout(function(){
      add(mk('position:fixed;top:-60px;left:'+(2+idx*2.9)+'%;font-size:'+(16+Math.random()*16)+'px;'+
        'z-index:9980;pointer-events:none;animation:brickFall '+(1.6+Math.random()*2)+'s linear forwards;opacity:.9',
        ICONS[Math.floor(Math.random()*ICONS.length)]));},idx*60);})(i);}
    var msgs=['📐 คำนวณราคาค่าก่อสร้าง 5,313 รายการ...','🧱 เทคอนกรีต ก่ออิฐ...','🏗️ ยกโครงสร้าง...',
      '👷 ตรวจรับงวดงาน...','🔨 เก็บงานละเอียด...','🎨 ทาสี ติดตั้งระบบ...','🏠 สร้างเสร็จตามงบ!'];
    msgs.forEach(function(m,i){setTimeout(function(){
      var t=add(mk('position:fixed;top:'+(8+i*9)+'%;right:24px;z-index:9993;pointer-events:none;'+
        'background:linear-gradient(90deg,#3d405b,#bc6c25);color:#fff;font-family:"Sarabun",sans-serif;'+
        'font-size:13px;padding:8px 18px;border-radius:20px;box-shadow:0 2px 12px rgba(188,108,37,.5)',m));
      t.style.transform='translateX(120%)';t.style.transition='transform .4s cubic-bezier(.17,.67,.35,1.3)';
      setTimeout(function(){t.style.transform='translateX(0)';},20);},400+i*350);});
    ['🚜','🚛','🏗️','🚧','🚛'].forEach(function(em,i){setTimeout(function(){
      add(mk('position:fixed;bottom:'+(6+i*7)+'%;left:-90px;font-size:'+(30+Math.random()*18)+'px;'+
        'z-index:9985;pointer-events:none;animation:truckRun '+(1.4+Math.random()*1.1)+'s linear forwards',em));},2200+i*260);});
    setTimeout(function(){add(mk('position:fixed;inset:0;background:#bc6c25;z-index:9970;pointer-events:none;animation:amberTint 2s ease forwards'));},3200);
    setTimeout(function(){
      var box=add(mk('position:fixed;top:50%;left:50%;z-index:10000;pointer-events:none;text-align:center;'+
        'background:linear-gradient(135deg,#283618 0%,#3d405b 50%,#bc6c25 100%);color:white;font-size:26px;font-weight:900;'+
        'padding:32px 52px;border-radius:28px;animation:popIn .65s cubic-bezier(.17,.67,.35,1.3) forwards,glowPulse 1s ease .7s infinite;'+
        'font-family:"Sarabun",sans-serif;line-height:1.7;min-width:360px',
        '🏗️ ก่อสร้างเสร็จสมบูรณ์! 🏗️<br><span style="font-size:17px">ประเมิน 69 ประเภทอาคาร · 77 จังหวัด</span><br>'+
        '<span style="font-size:15px;opacity:.85">รับเหมาตรงงบ ไม่บานปลาย 👷</span><br>'+
        '<span style="font-size:12px;opacity:.6">ตรวจราคาก่อนสร้าง ปลอดภัยกว่า 🧱</span>'));
      setTimeout(function(){box.style.animation='fadeOut .8s ease forwards';
        setTimeout(function(){trash.forEach(function(e){try{e.parentNode.removeChild(e);}catch(x){}});busy=false;},800);},3500);
    },4200);
  }
  window.addEventListener('load',function(){
    var btn=mk('position:fixed;bottom:28px;right:28px;font-size:46px;cursor:pointer;z-index:8000;'+
      'filter:drop-shadow(0 4px 12px rgba(188,108,37,.5));transition:transform .15s;user-select:none;line-height:1','🏗️');
    btn.addEventListener('click',go);
    btn.addEventListener('mouseenter',function(){btn.style.transform='scale(1.25) rotate(-10deg)';});
    btn.addEventListener('mouseleave',function(){btn.style.transform='';});
    document.body.appendChild(btn);
    var tip=mk('position:fixed;bottom:84px;right:28px;background:#212529dd;color:white;font-size:12px;padding:5px 11px;'+
      'border-radius:8px;z-index:8000;opacity:0;transition:opacity .2s;pointer-events:none;white-space:nowrap;font-family:Sarabun,sans-serif','กดเพื่อเริ่มก่อสร้าง! 👷');
    document.body.appendChild(tip);
    btn.addEventListener('mouseenter',function(){tip.style.opacity='1';});
    btn.addEventListener('mouseleave',function(){tip.style.opacity='0';});
  });
})();
</script>"""

html = f"""<!DOCTYPE html>
<html lang="th"><head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>ราคาค่าก่อสร้างอาคาร รายจังหวัด</title>
<link href="https://fonts.googleapis.com/css2?family=Sarabun:wght@400;600;700&display=swap" rel="stylesheet">
<style>{CSS}</style>
</head><body><div class="page">
<div class="hero">
  <h1>🏗️ ราคามาตรฐานค่าก่อสร้างอาคาร รายจังหวัด</h1>
  <p>วิเคราะห์ราคาค่าก่อสร้างต่อตารางเมตร · 69 ประเภทอาคาร × 77 จังหวัด · Python, Plotly &amp; Scikit-learn</p>
  <span class="badge">{n_types} ประเภทอาคาร</span>
  <span class="badge">{n_prov} จังหวัด</span>
  <span class="badge">เฉลี่ย {avg_p:,.0f} ฿/ตร.ม.</span>
  <span class="badge">🔮 Ridge R²={r2:.3f}</span>
</div>
<div class="stats">
  <div class="stat-card" style="--ac:#bc6c25"><div class="num">{n_types}</div><div class="lbl">ประเภทอาคาร</div></div>
  <div class="stat-card" style="--ac:#3d405b"><div class="num">{n_prov}</div><div class="lbl">จังหวัด (ครบทั้งประเทศ)</div></div>
  <div class="stat-card" style="--ac:#e07a5f"><div class="num">{avg_p:,.0f}</div><div class="lbl">ราคาเฉลี่ย (฿/ตร.ม.)</div></div>
  <div class="stat-card" style="--ac:#606c38"><div class="num">{min_p:,.0f}–{max_p:,.0f}</div><div class="lbl">ช่วงราคา (฿/ตร.ม.)</div></div>
</div>
<div class="note">📐 ราคาค่าก่อสร้างมาตรฐานต่อตารางเมตร แยกตามประเภทอาคารและจังหวัด — ใช้เป็นฐานประเมินราคา/ภาษี · เพื่อการศึกษา/วิเคราะห์</div>
<div class="card"><div class="card-header">Treemap — หมวดอาคาร → ประเภท (ตามราคา)</div>
  {to_html(fig_tree, full_html=False, include_plotlyjs=True)}</div>
<div class="card"><div class="card-header">🗺️ แผนที่ — ราคาค่าก่อสร้างเฉลี่ยรายจังหวัด</div>
  {to_html(fig_map, full_html=False, include_plotlyjs=False)}</div>
<div class="grid-2">
  <div class="card"><div class="card-header">การกระจายราคาตามหมวด</div>
    {to_html(fig_box, full_html=False, include_plotlyjs=False)}</div>
  <div class="card"><div class="card-header">Top 15 อาคารแพงสุด</div>
    {to_html(fig_bar, full_html=False, include_plotlyjs=False)}</div>
</div>
<div class="grid-2">
  <div class="card"><div class="card-header">K-Means — PCA Projection</div>
    {to_html(fig_pca, full_html=False, include_plotlyjs=False)}</div>
  <div class="card"><div class="card-header">Cluster Profiles</div>
    {to_html(fig_prof, full_html=False, include_plotlyjs=False)}</div>
</div>
<div class="card"><div class="card-header">🔮 Predictive Model — Ridge Regression (Predicted vs Actual)</div>
  {to_html(fig_reg, full_html=False, include_plotlyjs=False)}</div>
</div>
{BUILD_JS}
</body></html>"""

with open('construct_viz.html','w',encoding='utf-8') as f:
    f.write(html)
print(f'Saved: construct_viz.html ({os.path.getsize("construct_viz.html")//1024} KB)')